In [1]:
import os
import numpy as np
import xarray as xr

import cartopy.crs as ccrs
import matplotlib.pyplot as plt

from json import loads, dumps
from pyproj import CRS
from pyproj.crs.datum import CustomDatum, CustomPrimeMeridian
from rioxarray.merge import merge_arrays, merge_datasets

In [2]:
def new_crs():
    crs = CRS.from_epsg(4326).to_json()
    crs_json = loads(crs)

    datum = CustomDatum(prime_meridian=CustomPrimeMeridian(18)).to_json()
    datum_json = loads(datum)
    
    custom_crs_json = crs_json
    custom_crs_json['datum'] = datum_json
    custom_crs_json = dumps(custom_crs_json)
    custom_crs = CRS.from_json(custom_crs_json)
    
    return custom_crs

In [3]:
path = './weather/2021/'

files = [path + fname 
         for fname in np.sort(os.listdir(path)) 
         if fname.endswith('.nc')]

In [4]:
data = []
new_crs = new_crs()

for file in files:
    weather = xr.open_dataset(file)
    weather = weather.rio.write_crs(new_crs)
    weather = weather.rio.set_spatial_dims(x_dim='x', y_dim='y')

    data.append(weather)

In [5]:
ds = xr.merge(data)
ds['Month'] = (['time'], ds.time.dt.month.values)
ds

<xarray.Dataset>
Dimensions:                      (time: 184, x: 1721, y: 411)
Coordinates:
  * time                         (time) datetime64[ns] 2021-05-01 ... 2021-10-31
  * x                            (x) float64 1.1 1.2 1.3 ... 172.9 173.0 173.1
  * y                            (y) float64 81.9 81.8 81.7 ... 41.1 41.0 40.9
    spatial_ref                  int64 0
Data variables:
    Relative_Humidity_min        (time, y, x) float32 ...
    Precipitation_Flux           (time, y, x) float32 ...
    Temperature_Air_2m_Mean_24h  (time, y, x) float32 ...
    Wind_Speed_10m_Mean          (time, y, x) float32 ...
    Month                        (time) int64 5 5 5 5 5 5 ... 10 10 10 10 10 10
Attributes:
    Conventions:  CF-1.7

In [6]:
variables = list(ds.data_vars)  

for var in variables:
    if 'grid_mapping' in ds[var].attrs.keys():
        del ds[var].attrs['grid_mapping']

In [7]:
ds.to_netcdf('fwi/data/2021.nc')